# <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;  " > 实验：基于内容过滤的深度学习

在本练习中，你将使用神经网络实现基于内容的过滤，以构建一个电影推荐系统。


# 大纲
- [ 1 - 软件包 ](#1)
- [ 2 - 电影评分数据集 ](#2)
- [ 3 - 使用神经网络进行基于内容的过滤](#3)
  - [ 3.1 训练数据](#3.1)
  - [ 3.2 准备训练数据](#3.2)
- [ 4 - 用于基于内容过滤的神经网络](#4)
  - [ 练习 1](#ex01)
- [ 5 - 预测](#5)
  - [ 5.1 - 为新用户预测](#5.1)
  - [ 5.2 - 为已有用户预测](#5.2)
  - [ 5.3 - 查找相似项目](#5.3)
    - [ 练习 2](#ex02)
- [ 6 - 恭喜！](#6)


_**注意：** 为防止自动评分器出错，你不允许编辑或删除本实验中的非评分单元格。请也不要添加任何新单元格。
**通过本作业后**，如果你想尝试任何非评分代码，可以按照本笔记本底部的说明进行操作。_

<a name="1"></a>
## 1 - 软件包 <img align="left" src="./images/movie_camera.png"     style=" width:40px;  ">
我们将使用熟悉的软件包 NumPy、TensorFlow 以及 [scikit-learn](https://scikit-learn.org/stable/) 中的实用例程。我们还将使用 [tabulate](https://pypi.org/project/tabulate/) 整齐地打印表格，使用 [Pandas](https://pandas.pydata.org/) 组织表格数据。

In [2]:
import numpy as np
import numpy.ma as ma
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
pd.set_option("display.precision", 1)

<a name="2"></a>
## 2 - 电影评分数据集 <img align="left" src="./images/film_rating.png" style=" width:40px;" >
该数据集源自 [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) 数据集。

[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集包含大约 9000 部电影，由 600 位用户评分，评分范围为 0.5 到 5，步长为 0.5。数据集已缩减大小，专注于 2000 年以来的电影和热门类型。缩减后的数据集包含 $n_u = 397$ 位用户、$n_m= 847$ 部电影和 25521 条评分。每部电影提供电影名称、上映日期和一个或多个类型。例如 "Toy Story 3" 于 2010 年上映，有多个类型："Adventure|Animation|Children|Comedy|Fantasy"。该数据集除了评分外，几乎不包含用户信息。该数据集用于为下文描述的神经网络创建训练向量。
让我们进一步了解这个数据集。下表显示了按评分数量排名的前 10 部电影。这些电影恰好也有较高的平均评分。你看过其中几部？

In [3]:
top10_df = pd.read_csv("./data/content_top10_df.csv")
bygenre_df = pd.read_csv("./data/content_bygenre_df.csv")
top10_df

,movie id,num ratings,ave rating,title,genres
0,4993,198,4.1,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
1,5952,188,4.0,"Lord of the Rings: The Two Towers, The",Adventure|Fantasy
2,7153,185,4.1,"Lord of the Rings: The Return of the King, The",Action|Adventure|Drama|Fantasy
3,4306,170,3.9,Shrek,Adventure|Animation|Children|Comedy|Fantasy|Ro...
4,58559,149,4.2,"Dark Knight, The",Action|Crime|Drama
5,6539,149,3.8,Pirates of the Caribbean: The Curse of the Bla...,Action|Adventure|Comedy|Fantasy
6,79132,143,4.1,Inception,Action|Crime|Drama|Mystery|Sci-Fi|Thriller
7,6377,141,4.0,Finding Nemo,Adventure|Animation|Children|Comedy
8,4886,132,3.9,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy
9,7361,131,4.2,Eternal Sunshine of the Spotless Mind,Drama|Romance|Sci-Fi


下表显示了按类型排序的信息。每个类型的评分数量差异很大。请注意，一部电影可能有多个类型，因此下面的评分总和大于原始评分数量。

In [4]:
bygenre_df

,genre,num movies,ave rating/genre,ratings per genre
0,Action,321,3.4,10377
1,Adventure,234,3.4,8785
2,Animation,76,3.6,2588
3,Children,69,3.4,2472
4,Comedy,326,3.4,8911
5,Crime,139,3.5,4671
6,Documentary,13,3.8,280
7,Drama,342,3.6,10201
8,Fantasy,124,3.4,4468
9,Horror,56,3.2,1345


<a name="3"></a>
## 3 - 使用神经网络进行基于内容的过滤

在协同过滤实验中，你生成了两个向量：用户向量和电影/项目向量，它们的点积用于预测评分。这些向量仅从评分中推导而来。

基于内容的过滤也会生成用户和电影特征向量，但它认识到可能有关于用户和/或电影的其他可用信息可以改善预测。这些额外信息被提供给神经网络，然后神经网络生成用户和电影向量，如下图所示。
<figure>
    <center> <img src="./images/RecSysNN.png"   style="width:500px;height:280px;" ></center>
</figure>

<a name="3.1"></a>
### 3.1 训练数据
提供给网络的电影内容是原始数据和一些"工程特征"的组合。回顾第 1 课程第 2 周实验 4 中的特征工程讨论和实验。原始特征是电影上映年份和电影类型（以独热向量表示）。共有 14 种类型。工程特征是从用户评分中推导出的平均评分。

用户内容由工程特征组成。为每位用户计算每种类型的平均评分。此外，用户 ID、评分数量和评分平均值也可用，但不包含在训练或预测内容中。它们随数据集一起保留，因为它们在解释数据时很有用。

训练集由数据集中所有用户给出的评分组成。一些评分被重复，以增加代表性不足的类型的训练样本数量。训练集被分成两个具有相同条目数的数组：用户数组和电影/项目数组。

下面让我们加载并显示一些数据。

In [5]:
# 加载数据，设置配置变量
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # 在训练期间移除用户 ID、评分数量和平均评分
num_item_features = item_train.shape[1] - 1  # 在训练时移除电影 ID
uvs = 3  # 用户类型向量起始位置
ivs = 3  # 项目类型向量起始位置
u_s = 3  # 训练中使用的列起始位置，用户
i_s = 1  # 训练中使用的列起始位置，项目
print(f"Number of training vectors: {len(item_train)}")

Number of training vectors: 50884


让我们看看用户训练数组的前几个条目。

In [6]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9


部分用户和项目/电影特征不用于训练。在上表中，括号 "[]" 中的特征（如 "user id"、"rating count" 和 "rating ave"）在模型训练和使用时不包含在内。
上面你可以看到用户 2 的每种类型平均评分。零条目是用户未评分的类型。用户向量对用户评分的所有电影都是相同的。
让我们看看电影/项目数组的前几个条目。

In [7]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

[movie id],year,ave rating,Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
6874,2003,4.0,1,0,0,0,0,1,0,0,0,0,0,0,0,1
8798,2004,3.8,1,0,0,0,0,1,0,1,0,0,0,0,0,1
46970,2006,3.2,1,0,0,0,1,0,0,0,0,0,0,0,0,0
48516,2006,4.3,0,0,0,0,0,1,0,1,0,0,0,0,0,1
58559,2008,4.2,1,0,0,0,0,1,0,1,0,0,0,0,0,0


上面，电影数组包含电影上映年份、平均评分以及每种潜在类型的指示符。对于适用于该电影的类型，指示符为 1。电影 ID 不用于训练，但在解释数据时很有用。

In [8]:
print(f"y_train[:5]: {y_train[:5]}")

y_train[:5]: [4.  3.5 4.  4.  4.5]


目标 y 是用户给出的电影评分。

上面我们可以看到电影 6874 是一部 2003 年上映的 Action/Crime/Thriller 电影。用户 2 对动作电影的平均评分为 3.9。MovieLens 用户对该电影的平均评分为 4。"y" 为 4，表示用户 2 也将电影 6874 评分为 4。一个训练样本由用户和项目数组中的一行以及 y_train 中的一个评分组成。

<a name="3.2"></a>
### 3.2 准备训练数据
回顾第 1 课程第 2 周，你探索了特征缩放作为改善收敛的手段。我们将使用 [scikit learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) 对输入特征进行缩放。这在第 1 课程第 2 周实验 5 中使用过。下面还展示了 inverse_transform 以产生原始输入。我们将使用 Min Max Scaler 对目标评分进行缩放，将目标缩放到 -1 和 1 之间。[scikit learn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

In [9]:
# 缩放训练数据
item_train_unscaled = item_train
user_train_unscaled = user_train
y_train_unscaled    = y_train

scalerItem = StandardScaler()
scalerItem.fit(item_train)
item_train = scalerItem.transform(item_train)

scalerUser = StandardScaler()
scalerUser.fit(user_train)
user_train = scalerUser.transform(user_train)

scalerTarget = MinMaxScaler((-1, 1))
scalerTarget.fit(y_train.reshape(-1, 1))
y_train = scalerTarget.transform(y_train.reshape(-1, 1))
#ynorm_test = scalerTarget.transform(y_test.reshape(-1, 1))

print(np.allclose(item_train_unscaled, scalerItem.inverse_transform(item_train)))
print(np.allclose(user_train_unscaled, scalerUser.inverse_transform(user_train)))

True
True


为了评估结果，我们将按照第 2 课程第 3 周的讨论，将数据分为训练集和测试集。这里我们将使用 [sklean train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 来分割和打乱数据。请注意，将初始随机状态设置为相同的值可确保项目、用户和 y 以相同方式被打乱。

In [10]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test data shape: {item_test.shape}")

movie/item training data shape: (40707, 17)
movie/item test data shape: (10177, 17)


经过缩放和打乱的数据现在的均值为零。

In [11]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
1,0,-1.0,-0.8,-0.7,0.1,-0.0,-1.2,-0.4,0.6,-0.5,-0.5,-0.1,-0.6,-0.6,-0.7,-0.7
0,1,-0.7,-0.5,-0.7,-0.1,-0.2,-0.6,-0.2,0.7,-0.5,-0.8,0.1,-0.0,-0.6,-0.5,-0.4
-1,-1,-0.2,0.3,-0.4,0.4,0.5,1.0,0.6,-1.2,-0.3,-0.6,-2.3,-0.1,0.0,0.4,-0.0
0,-1,0.6,0.5,0.5,0.2,0.6,-0.1,0.5,-1.2,0.9,1.2,-2.3,-0.1,0.0,0.2,0.3
-1,0,0.7,0.6,0.5,0.3,0.5,0.4,0.6,1.0,0.6,0.3,0.8,0.8,0.4,0.7,0.7


<a name="4"></a>
## 4 - 用于基于内容过滤的神经网络
现在，让我们按照上图描述构建一个神经网络。它将有两个通过点积组合的网络。你将构建这两个网络。在本例中，它们是相同的。请注意，这些网络不需要相同。如果用户内容比电影内容大得多，你可能会选择增加用户网络相对于电影网络的复杂度。在本例中，内容相似，因此网络相同。

<a name="ex01"></a>
### 练习 1

- 使用 Keras 顺序模型
    - 第一层是具有 256 个单元和 relu 激活的全连接层。
    - 第二层是具有 128 个单元和 relu 激活的全连接层。
    - 第三层是具有 `num_outputs` 个单元和线性（或无）激活的全连接层。
    
网络的其余部分将被提供。提供的代码不使用 Keras 顺序模型，而是使用 Keras [函数式 API](https://keras.io/guides/functional_api/)。这种格式在组件互连方式上提供了更大的灵活性。


In [12]:
# 评分单元格
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### 开始编写代码 ###     
      tf.keras.layers.Dense(units = 256, activation = "relu"),
      tf.keras.layers.Dense(units = 128, activation = "relu"),
      tf.keras.layers.Dense(units = num_outputs)
    ### 结束编写代码 ###  
])

item_NN = tf.keras.models.Sequential([
    ### 开始编写代码 ###     
      tf.keras.layers.Dense(units = 256, activation = "relu"),
      tf.keras.layers.Dense(units = 128, activation = "relu"),
      tf.keras.layers.Dense(units = num_outputs)
    ### 结束编写代码 ###  
])

# 创建用户输入并指向基础网络
input_user = tf.keras.layers.Input(shape=(num_user_features,))
vu = user_NN(input_user)
vu = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(vu)

# 创建项目输入并指向基础网络
input_item = tf.keras.layers.Input(shape=(num_item_features,))
vm = item_NN(input_item)
vm = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(vm)

# 计算两个向量 vu 和 vm 的点积
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# 指定模型的输入和输出
model = tf.keras.Model([input_user, input_item], output)

model.summary()

W0000 00:00:1785939749.199394   65384 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 14)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32)        │     40,864 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 32)        │     41,376 │ input_layer_2[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 32)        │          0 │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 32)        │          0 │ sequential_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ lambda[0][0],     │
│                     │                   │            │ lambda_1[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 82,240 (321.25 KB)

 Trainable params: 82,240 (321.25 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# 公共测试
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

所有测试通过！
所有测试通过！


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  你可以按如下方式创建一个带有 relu 激活的全连接层。
    
```python     
user_NN = tf.keras.models.Sequential([
    ### 开始编写代码 ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### 结束编写代码 ###  
])

item_NN = tf.keras.models.Sequential([
    ### 开始编写代码 ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### 结束编写代码 ###  
])
```    
<details>
    <summary><font size="2" color="darkblue"><b> 点击查看答案</b></font></summary>
    
```python 
user_NN = tf.keras.models.Sequential([
    ### 开始编写代码 ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### 结束编写代码 ###  
])

item_NN = tf.keras.models.Sequential([
    ### 开始编写代码 ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### 结束编写代码 ###  
])
```
</details>
</details>

    


我们将使用均方误差损失和 Adam 优化器。

In [14]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [15]:
tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], y_train, epochs=30)

Epoch 1/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1235
Epoch 2/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1143
Epoch 3/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1093
Epoch 4/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1060
Epoch 5/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1033
Epoch 6/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1005
Epoch 7/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0980
Epoch 8/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0956
Epoch 9/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0936
Epoch 10/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0917
Epoch 11/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0900
Epoch 12/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0885
Epoch 13/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0871
Epoch 14/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0858
Epoch 15/30
1273/1273 ━━━━━━━

评估模型以确定测试数据上的损失。

In [16]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], y_test)

319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0839


0.08386974036693573

它与训练损失相当，表明模型没有对训练数据过度拟合。

<a name="5"></a>
## 5 - 预测
下面，你将在多种情况下使用模型进行预测。
<a name="5.1"></a>
### 5.1 - 为新用户预测
首先，我们将创建一个新用户，并让模型为该用户推荐电影。在示例用户内容上尝试后，你可以自由更改用户内容以匹配你自己的偏好，看看模型推荐什么。请注意，评分在 0.5 到 5.0 之间（含端点），步长为 0.5。

In [17]:
new_user_id = 5000
new_rating_ave = 0.0
new_action = 0.0
new_adventure = 5.0
new_animation = 0.0
new_childrens = 0.0
new_comedy = 0.0
new_crime = 0.0
new_documentary = 0.0
new_drama = 0.0
new_fantasy = 5.0
new_horror = 0.0
new_mystery = 0.0
new_romance = 0.0
new_scifi = 0.0
new_thriller = 0.0
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

新用户喜欢冒险和奇幻类型的电影。让我们为新用户查找评分最高的电影。
下面，我们将使用一组电影/项目向量 `item_vecs`，它包含训练/测试集中每部电影的向量。将其与上面的新用户向量匹配，使用缩放后的向量预测所有电影的评分。

In [18]:
# 生成并复制用户向量以匹配数据集中的电影数量。
user_vecs = gen_user_vecs(user_vec,len(item_vecs))

# 缩放用户和项目向量
suser_vecs = scalerUser.transform(user_vecs)
sitem_vecs = scalerItem.transform(item_vecs)

# 进行预测
y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

# 反缩放 y 预测
y_pu = scalerTarget.inverse_transform(y_p)

# 对结果排序，最高预测在前
sorted_index = np.argsort(-y_pu,axis=0).reshape(-1).tolist()  # 取反以获得最大评分在前
sorted_ypu   = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index]  # 使用未缩放的向量进行显示

print_pred_movies(sorted_ypu, sorted_items, movie_dict, maxcount = 10)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


y_p,movie id,rating ave,title,genres
4.4,98809,3.8,"Hobbit: An Unexpected Journey, The (2012)",Adventure|Fantasy
4.3,40815,3.8,Harry Potter and the Goblet of Fire (2005),Adventure|Fantasy|Thriller
4.2,5816,3.6,Harry Potter and the Chamber of Secrets (2002),Adventure|Fantasy
4.2,59501,3.5,"Chronicles of Narnia: Prince Caspian, The (2008)",Adventure|Children|Fantasy
4.2,106489,3.6,"Hobbit: The Desolation of Smaug, The (2013)",Adventure|Fantasy
4.2,81834,4,Harry Potter and the Deathly Hallows: Part 1 (2010),Action|Adventure|Fantasy
4.2,4896,3.8,Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001),Adventure|Children|Fantasy
4.2,8368,3.9,Harry Potter and the Prisoner of Azkaban (2004),Adventure|Fantasy
4.1,54001,3.9,Harry Potter and the Order of the Phoenix (2007),Adventure|Drama|Fantasy
4.1,41566,3.4,"Chronicles of Narnia: The Lion, the Witch and the Wardrobe, The (2005)",Adventure|Children|Fantasy


<a name="5.2"></a>
### 5.2 - 为已有用户预测
让我们看看对数据集中用户 "user 2" 的预测。我们可以将预测评分与模型的评分进行比较。

In [19]:
uid = 2 
# 组建一组用户向量。这是同一个向量，经过变换和重复。
user_vecs, y_vecs = get_user_vecs(uid, user_train_unscaled, item_vecs, user_to_genre)

# 缩放用户和项目向量
suser_vecs = scalerUser.transform(user_vecs)
sitem_vecs = scalerItem.transform(item_vecs)

# 进行预测
y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

# 反缩放 y 预测
y_pu = scalerTarget.inverse_transform(y_p)

# 对结果排序，最高预测在前
sorted_index = np.argsort(-y_pu,axis=0).reshape(-1).tolist()  # 取反以获得最大评分在前
sorted_ypu   = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index]  # 使用未缩放的向量进行显示
sorted_user  = user_vecs[sorted_index]
sorted_y     = y_vecs[sorted_index]

# 打印用户已评分电影的排序预测结果
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, ivs, uvs, movie_dict, maxcount = 50)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


y_p,y,user,user genre ave,movie rating ave,movie id,title,genres
4.6,5.0,2,[4.0],4.3,80906,Inside Job (2010),Documentary
4.4,3.5,2,"[4.0,4.1,4.0,3.9]",3.8,8798,Collateral (2004),Action|Crime|Drama|Thriller
4.4,4.0,2,"[4.0,4.1,3.9]",4.0,6874,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller
4.3,4.5,2,"[4.0,4.0]",4.1,68157,Inglourious Basterds (2009),Action|Drama
4.2,4.0,2,"[4.0,4.1,4.0,4.0,3.9,3.9]",4.1,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller
4.2,4.0,2,"[4.1,4.0,3.9]",4.3,48516,"Departed, The (2006)",Crime|Drama|Thriller
4.2,3.5,2,"[4.0,4.0]",3.9,99114,Django Unchained (2012),Action|Drama
4.2,4.5,2,"[4.0,4.1,4.0]",4.2,58559,"Dark Knight, The (2008)",Action|Crime|Drama
4.1,5.0,2,"[4.0,4.1,4.0]",3.9,106782,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama
4.0,4.0,2,"[4.0,4.0,3.9]",4.0,74458,Shutter Island (2010),Drama|Mystery|Thriller


模型预测通常在实际评分的 1 分以内，尽管它对用户对特定电影的评分预测不是很准确。如果用户的评分与其类型平均评分有显著差异，情况尤其如此。你可以在上面更改用户 ID 以尝试不同的用户。并非所有用户 ID 都在训练集中使用过。

<a name="5.3"></a>
### 5.3 - 查找相似项目
上面的神经网络产生两个特征向量：用户特征向量 $v_u$ 和电影特征向量 $v_m$。这些是 32 维向量，其值难以解释。然而，相似的项目会有相似的向量。这些信息可用于做出推荐。例如，如果一位用户对 "Toy Story 3" 给出了高评分，可以通过选择具有相似电影特征向量的电影来推荐类似的电影。

相似性度量是两个向量 $ \mathbf{v_m^{(k)}}$ 和 $\mathbf{v_m^{(i)}}$ 之间的平方距离：
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$

<a name="ex02"></a>
### 练习 2

编写一个函数来计算平方距离。

In [20]:
# 评分函数：sq_dist
# UNQ_C2
def sq_dist(a,b):
    """
    返回两个向量之间的平方距离
    参数：
      a (ndarray (n,)): 具有 n 个特征的向量
      b (ndarray (n,)): 具有 n 个特征的向量
    返回：
      d (float) : 距离
    """
    ### 开始编写代码 ###     
    d = np.sum(np.square(a - b))
    ### 结束编写代码 ###     
    return d

In [21]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1):0.3f}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2):0.3f}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3):0.3f}")

squared distance between a1 and b1: 0.000
squared distance between a2 and b2: 0.030
squared distance between a3 and b3: 2.000


**期望输出**：

squared distance between a1 and b1: 0.000    
squared distance between a2 and b2: 0.030   
squared distance between a3 and b3: 2.000

In [22]:
# 公共测试
test_sq_dist(sq_dist)

所有测试通过！


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  虽然求和通常表示应使用 for 循环，但这里减法可以在一条语句中逐元素进行。此外，你可以利用 np.square 对减法结果逐元素求平方。np.sum 可用于对平方后的元素求和。
    
</details>

    


电影之间的距离矩阵可以在模型训练后一次性计算，然后在不重新训练的情况下用于新的推荐。模型训练后的第一步是获取每部电影的电影特征向量 $v_m$。为此，我们将使用训练好的 `item_NN` 并构建一个小模型，以便我们可以将电影向量通过它来生成 $v_m$。

In [23]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features,))    # 输入层
vm_m = item_NN(input_item_m)                                       # 使用训练好的 item_NN
vm_m = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(vm_m)  # 将归一化纳入模型，与原始模型一致
model_m = tf.keras.Model(input_item_m, vm_m)                                
model_m.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 32)             │        41,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_2 (Lambda)               │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,376 (161.62 KB)

 Trainable params: 41,376 (161.62 KB)

 Non-trainable params: 0 (0.00 B)

有了电影模型后，你可以使用模型对一组项目/电影向量作为输入进行预测，从而创建一组电影特征向量。`item_vecs` 是所有电影向量的集合。它必须经过缩放才能与训练好的模型一起使用。预测结果是每部电影的 32 维特征向量。

In [24]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
size of all predicted movie feature vectors: (847, 32)


现在让我们计算每部电影特征向量与所有其他电影特征向量之间的平方距离矩阵：
<figure>
    <left> <img src="./images/distmatrix.PNG"   style="width:400px;height:225px;" ></center>
</figure>

然后我们可以通过沿每行查找最小值来找到最接近的电影。我们将利用 [numpy masked arrays](https://numpy.org/doc/1.21/user/tutorial-ma.html) 来避免选择同一部电影。对角线上的掩码值不会包含在计算中。

In [25]:
count = 50  # 要显示的电影数量
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # 掩码对角线

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    disp.append( [movie_dict[movie1_id]['title'], movie_dict[movie1_id]['genres'],
                  movie_dict[movie2_id]['title'], movie_dict[movie1_id]['genres']]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow")
table

movie1,genres,movie2,genres
Save the Last Dance (2001),Drama|Romance,Mona Lisa Smile (2003),Drama|Romance
"Wedding Planner, The (2001)",Comedy|Romance,Mr. Deeds (2002),Comedy|Romance
Hannibal (2001),Horror|Thriller,Final Destination 2 (2003),Horror|Thriller
Saving Silverman (Evil Woman) (2001),Comedy|Romance,"Sweetest Thing, The (2002)",Comedy|Romance
Down to Earth (2001),Comedy|Fantasy|Romance,Bewitched (2005),Comedy|Fantasy|Romance
"Mexican, The (2001)",Action|Comedy,Rush Hour 2 (2001),Action|Comedy
15 Minutes (2001),Thriller,Panic Room (2002),Thriller
Enemy at the Gates (2001),Drama,"Aviator, The (2004)",Drama
Heartbreakers (2001),Comedy|Crime|Romance,America's Sweethearts (2001),Comedy|Crime|Romance
Spy Kids (2001),Action|Adventure|Children|Comedy,Scooby-Doo (2002),Action|Adventure|Children|Comedy


结果表明，模型通常会推荐具有相似类型的电影。

<a name="6"></a>
## 6 - 恭喜！<img align="left" src="./images/film_award.png" style=" width:40px;">
你已完成基于内容的推荐系统。

这种结构是许多商业推荐系统的基础。如果可用，用户内容可以大幅扩展以包含更多关于用户的信息。项目不限于电影。这可以用于推荐任何项目：书籍、汽车或与你"购物车"中项目相似的物品。

<details>
  <summary><font size="2" color="darkgreen"><b>如果你想尝试任何非评分代码，请点击此处。</b></font></summary>
    <p><i><b>重要提示：请仅在你已通过作业后才执行此操作，以避免自动评分器出现问题。</b></i>
    <ol>
        <li> 在笔记本的菜单中，点击 "View" > "Cell Toolbar" > "Edit Metadata"</li>
        <li> 点击你想要锁定/解锁的代码单元格旁边的 "Edit Metadata" 按钮</li>
        <li> 将 "editable" 的属性值设置为：
            <ul>
                <li> "true" 如果你想解锁它</li>
                <li> "false" 如果你想锁定它</li>
            </ul>
        </li>
        <li> 在笔记本的菜单中，点击 "View" > "Cell Toolbar" > "None"</li>
    </ol>
    <p> 以下是上述步骤的简短演示：
        <br>
        <img src="https://drive.google.com/uc?export=view&id=14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" align="center" alt="unlock_cells.gif">
</details>